# EDA: Swire Coca‑Cola

- Authors: Ali Ladha, Cyrus Sobhani, Robby Stohel, and Sterling LeDuc 
- Date: 2025-10-04




# Introduction

Cart abandonment represents a significant challenge for digital ordering platforms, directly affecting both revenue and customer retention. For MyCoke360, Coca-Cola’s B2B digital ordering platform launched in summer 2024 by Swire Coca-Cola (Swire), understanding the behavioral and operational factors that contribute to incomplete purchases is essential. The platform serves Foodservice On-Premise (FSOP) customers such as restaurants, schools, hospitals, and retailers, where order frequency, timing, and product mix are key drivers of business performance. This exploratory analysis combines Google Analytics behavioral data with order and sales records to identify the patterns and conditions under which carts are abandoned. The goal is to provide actionable insights that support strategic interventions to improve order completion rates and customer engagement.


## Business Problem

Swire experiences measurable revenue loss each order cycle due to cart abandonment, where customers add products to their carts but fail to submit orders by the designated cutoff. This behavior not only results in missed sales opportunities but also disrupts plant logistics, complicates demand forecasting, and weakens customer relationships within the FSOP channel. Addressing this issue requires a data-driven understanding of the underlying behaviors, temporal trends, and operational factors that influence abandonment, enabling Swire to design targeted solutions that reduce financial loss and enhance platform efficiency.



## Initial Guiding Questions

- Which behavioral events or sequences of events are the strongest predictors of cart abandonment?
- What specific behaviors or conditions lead customers to return and complete a previously abandoned cart?
- What actions typically occur after a cart is abandoned? Do customers complete their purchase through another channel, or do they churn?
- What is the financial impact of cart abandonment on overall MyCoke360 revenue and product mix?
- Which products appear most frequently in abandoned carts?
- How does cart abandonment vary by device type, and what strategies can reduce it?

While these guiding questions move closer to addressing the main problem of the project, the exploratory data analysis may not answer them directly. Instead, they will serve to guide how the data is structured, organized, and prepared so that later modeling and analysis can properly evaluate the impact of cart abandonment.

## Initial Setup

In [0]:
# Imports
from functools import reduce
from typing import Dict, List, Tuple

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

from pyspark.sql import DataFrame, Row
from pyspark.sql.functions import (
    coalesce,
    col,
    concat,
    count,
    countDistinct,
    count as count_,
    desc,
    expr,
    format_number,
    from_json,
    initcap,
    isnan,
    lit,
    lower,
    mean,
    min as min_,
    max as max_,
    regexp_replace,
    round as round_,
    size,
    stddev_samp,
    struct,
    sum as sum_,
    to_date,
    to_timestamp,
    transform,
    trim,
    when,
)
from pyspark.sql.types import (
    ArrayType,
    BooleanType,
    DoubleType,
    FloatType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

## Helper Functions

In [0]:
def read_csv_reformat(path: str) -> DataFrame:
    """Read a CSV with Spark and make columns snake_case and lowercase."""
    data = spark.read.csv(
        path,
        header=True,
        inferSchema=False,
        quote='"',
        escape='"',
        multiLine=True,
    )

    fixed_cols = [c.strip().lower().replace(" ", "_") for c in data.columns]
    return data.toDF(*fixed_cols)


def summarize(df: DataFrame) -> DataFrame:
    """Summarize a Spark DataFrame with stats and null-like counts."""
    # Get the standard summary rows
    summary_df = df.summary()
    sum_cols = [c for c in summary_df.columns if c != "summary"]

    # Compute null counts per column with type-aware emptiness
    exprs = []
    for column_name in df.columns:
        dt = df.schema[column_name].dataType
        is_empty_like = col(column_name).isNull()

        if isinstance(dt, StringType):
            is_empty_like = (
                is_empty_like |
                (col(column_name) == "") |
                (lower(col(column_name)) == "null")
            )

        if isinstance(dt, ArrayType):
            is_empty_like = is_empty_like | (size(col(column_name)) == 0)

        exprs.append(sum_(when(is_empty_like, 1).otherwise(0)).alias(column_name))

    null_counts = df.agg(*exprs).collect()[0].asDict()

    # Convert null_counts to a Spark DataFrame row
    null_counts_row = df.sparkSession.createDataFrame(
        [("nulls",) + tuple(str(null_counts[c]) for c in sum_cols)],
        ["summary"] + sum_cols
    )

    # Align columns and append the nulls row
    summary_df = summary_df.select(["summary"] + sum_cols)
    final_summary = summary_df.unionByName(null_counts_row)

    display(final_summary)
    return final_summary


def make_all_text_lowercase(dataframes: dict) -> dict:
    """Go through each Spark DataFrame and lowercase all string columns."""
    updated_dataframes = {}
    for df_name, df in dataframes.items():
        for column_name, column_type in df.dtypes:
            if column_type == "string":
                df = df.withColumn(column_name, lower(col(column_name)))
        updated_dataframes[df_name] = df
    return updated_dataframes



def datatype_and_normalization_check(datasets: dict, num_rows: int = 3) -> DataFrame:
    """Create a Spark DataFrame listing each table, column, and its first three values."""
    records = []

    for table_name, sdf in datasets.items():
        sample_rows = sdf.limit(num_rows).collect()
        if not sample_rows:
            # Fill with Nones if table is empty
            records.append(
                (table_name, None, *([None] * num_rows))
            )
            continue

        row_dicts = [row.asDict() for row in sample_rows]
        columns = sdf.columns

        for col_name in columns:
            # Collect up to num_rows sample values
            values = [
                str(row_dicts[i].get(col_name))
                if i < len(row_dicts) else None
                for i in range(num_rows)
            ]
            records.append((table_name, col_name, *values))

    # Build schema dynamically
    schema_fields = [
        StructField("table", StringType()),
        StructField("column", StringType())
    ] + [
        StructField(f"row_{i + 1}", StringType())
        for i in range(num_rows)
    ]

    schema = StructType(schema_fields)

    return spark.createDataFrame(records, schema)


def missing_check(tables: Dict[str, DataFrame]) -> DataFrame:
    spark = next(iter(tables.values())).sparkSession
    result_rows = []

    for table_name, df in tables.items():
        agg_exprs = []
        for field in df.schema.fields:
            name = field.name
            dtype = field.dataType

            agg_exprs.append(
                sum_(when(col(name).isNull(), 1).otherwise(0)).alias(f"{name}__null")
            )

            if isinstance(dtype, (FloatType, DoubleType)):
                agg_exprs.append(
                    sum_(when(isnan(col(name)), 1).otherwise(0)).alias(f"{name}__nan")
                )
            else:
                agg_exprs.append(sum_(lit(0)).alias(f"{name}__nan"))

            if isinstance(dtype, StringType):
                agg_exprs.append(
                    sum_(when(col(name) == "", 1).otherwise(0)).alias(f"{name}__empty")
                )
                agg_exprs.append(
                    sum_(when(lower(col(name)) == "null", 1).otherwise(0)).alias(
                        f"{name}__literal_null"
                    )
                )
            else:
                agg_exprs.append(sum_(lit(0)).alias(f"{name}__empty"))
                agg_exprs.append(sum_(lit(0)).alias(f"{name}__literal_null"))

            if isinstance(dtype, ArrayType):
                agg_exprs.append(
                    sum_(when(size(col(name)) == 0, 1).otherwise(0)).alias(
                        f"{name}__empty_array"
                    )
                )
            else:
                agg_exprs.append(sum_(lit(0)).alias(f"{name}__empty_array"))

        agg_exprs.append(count_(lit(1)).alias("__rows"))

        stats = df.agg(*agg_exprs).first().asDict()
        total_rows = int(stats["__rows"])

        for name in df.columns:
            c_null = int(stats[f"{name}__null"])
            c_nan = int(stats[f"{name}__nan"])
            c_empty = int(stats[f"{name}__empty"])
            c_literal = int(stats[f"{name}__literal_null"])
            c_empty_arr = int(stats[f"{name}__empty_array"])

            total_missing = c_null + c_nan + c_empty + c_literal + c_empty_arr
            missing_prop = (total_missing / total_rows) if total_rows else None

            result_rows.append(
                (
                    table_name,
                    name,
                    c_null,
                    c_nan,
                    c_empty,
                    c_literal,
                    c_empty_arr,
                    total_missing,
                    missing_prop,
                )
            )

    schema = StructType(
        [
            StructField("table", StringType()),
            StructField("column", StringType()),
            StructField("null_count", LongType()),
            StructField("nan_count", LongType()),
            StructField("empty_string_count", LongType()),
            StructField("literal_null_count", LongType()),
            StructField("empty_array_count", LongType()),
            StructField("total_missing", LongType()),
            StructField("missing_prop", DoubleType()),
        ]
    )
    return spark.createDataFrame(result_rows, schema)


def quick_list(df, column_name):
    display(
        df
        .select(column_name)
        .distinct()
        .orderBy(column_name)
    )

# Data Description

The dataset consists of eight CSV tables covering customer behavior, transactions, and supporting reference information. Three fact tables capture activity on MyCoke360: Google Analytics events (site visits, add/remove cart actions, purchases, and device/page details), Orders (materials ordered per customer, with order type and timestamps in both EST and UTC), and Sales (fulfilled transactions with pricing and profit measures). These are complemented by five dimension tables: Customer (account and channel attributes, sales office details), Cutoff Times (order cutoff policies by plant, office, and distribution mode), Material (product master data such as pack type, brand, flavor, and category), Operating Hours (current ordering frequency and anchor day/date by customer), and Visit Plan (historical anchor dates, frequencies, and sales office attributes). Collectively, these tables create a comprehensive view of both customer behavior and business processes adequetely enabling our analysis.

## Data Loading & Structure

In [0]:
# Fact tables
google_analytics = read_csv_reformat(
    "/Volumes/workspace/default/capstone_data/google_analytics.csv"
)
orders = read_csv_reformat(
    "/Volumes/workspace/default/capstone_data/orders.csv"
)
sales = read_csv_reformat(
    "/Volumes/workspace/default/capstone_data/sales.csv"
)

# Dimension tables
customers = read_csv_reformat(
    "/Volumes/workspace/default/capstone_data/customer.csv"
)
materials = read_csv_reformat(
    "/Volumes/workspace/default/capstone_data/material.csv"
)
operating_hours = read_csv_reformat(
    "/Volumes/workspace/default/capstone_data/operating_hours.csv"
)
visit_plan = read_csv_reformat(
    "/Volumes/workspace/default/capstone_data/visit_plan.csv"
)
cutoff_times = read_csv_reformat(
    "/Volumes/workspace/default/capstone_data/cutoff_times.csv"
)

datasets = {
    "google_analytics": google_analytics,
    "orders": orders,
    "sales": sales,
    "customers": customers,
    "materials": materials,
    "operating_hours": operating_hours,
    "visit_plan": visit_plan,
    "cutoff_times": cutoff_times,
}

In [0]:
for name, df in datasets.items():
    row_count = df.count()
    col_count = len(df.columns)
    print(f"{name.replace('_', ' ').title()}: {row_count} rows, {col_count} columns")
    summarize(df)

# Transformation & Cleaning

Cleaning the data is an essential step before performing the EDA. This helps prevent analysis on data that contains incorrect data types, missing values, or outliers.

The cleaning process will be organized by table, focusing on:

1. Data type assignment and validation
2. Identification of missing or incomplete data
3. Evaluation of data sparsity for dimensionality reduction and materiality
4. Visualizations highlighting key characteristics and distributions within each table
5. Brief notes summarizing data quality, cleaning actions, and results for each table


## Data Typing & Normalization

In [0]:
peak_df = datatype_and_normalization_check(datasets, num_rows=2)
display(peak_df)
del(peak_df)

# quick_list(datasets["google_analytics"], "event_name")
# quick_list(datasets["google_analytics"], "device_category")
# quick_list(datasets["google_analytics"], "device_mobile_brand_name")
# quick_list(datasets["google_analytics"], "device_operating_system")
# quick_list(datasets["google_analytics"], "event_page_name")
# quick_list(datasets["google_analytics"], "event_page_title")

In [0]:
items_schema = ArrayType(
    StructType([
        StructField("item_id", StringType()),
        StructField("quantity", StringType())
    ])
)

datasets["google_analytics"] = (
    datasets["google_analytics"]
    .withColumn("customer_id", col("customer_id").cast("int"))
    .withColumn("event_date", to_date("event_date", "yyyy-MM-dd"))
    .withColumn("event_timestamp_utc", to_timestamp("event_timestamp"))
    .withColumn(
        "items",
        from_json(col("items"), items_schema)
    )
    .withColumn(
        "items",
        when(
            size(expr("filter(`items`, x -> x.item_id != '(not set)')")) == 0,
            None
        ).otherwise(expr("filter(`items`, x -> x.item_id != '(not set)')"))
    )
    .withColumn(
        "device_mobile_brand_name",
        when(trim(lower(col("device_mobile_brand_name"))) == "null", None)
        .otherwise(col("device_mobile_brand_name"))
    )
    .withColumn(
        "event_page_name",
        when(trim(lower(col("event_page_name"))) == "null", None)
        .otherwise(col("event_page_name"))
    )
    .withColumn(
        "event_page_title",
        when(trim(lower(col("event_page_title"))) == "null", None)
        .otherwise(col("event_page_title"))
    )
    .withColumn("event_name", lower(col("event_name")))
    .withColumn("device_category", initcap(col("device_category")))
    .withColumn("event_page_name", initcap(col("event_page_name")))
    .withColumn("event_page_title", initcap(col("event_page_title")))
    .select(
        "customer_id",
        "event_timestamp_utc",
        "event_name",
        "device_category",
        "device_mobile_brand_name",
        "device_operating_system",
        "event_page_name",
        "event_page_title",
        "items"
    )
    .orderBy("customer_id", "event_timestamp_utc")
)
datasets["google_analytics"].printSchema()

In [0]:
# Order casting
datasets["orders"] = (
    datasets["orders"]
    .withColumn("customer_id", col("customer_id").cast("int"))
    .withColumn("material_id", col("material_id").cast("int"))
    .withColumn("order_quantity", col("order_quantity").cast("double"))
    .withColumn("created_date_utc", to_timestamp("created_date_utc"))
    .withColumn("order_type", lower(regexp_replace("order_type", " ", "_")))
    .withColumn(
        "plant_id",
        when(trim(lower(col("plant_id"))) == "null", None)
        .otherwise(col("plant_id"))
    )
    .drop("created_date_est")
)
datasets["orders"].printSchema()
# quick_list(datasets["orders"], "order_type")
# quick_list(datasets["orders"], "plant_id")

In [0]:
# Sale casting
datasets["sales"] = (
    datasets["sales"]
    .withColumn("posting_date", to_date("posting_date", "M/d/yyyy"))
    .withColumn("customer_id", col("customer_id").cast("int"))
    .withColumn("material_id", col("material_id").cast("int"))
    .withColumn("gross_profit_dead_net", col("gross_profit_dead_net").cast("double"))
    .withColumn("physical_volume", col("physical_volume").cast("double"))
    .withColumn("nsi_dead_net", col("nsi_dead_net").cast("double"))
)
datasets["sales"].printSchema()

In [0]:
# Customer casting
datasets["customers"] = (
    datasets["customers"]
)
datasets["customers"].printSchema()

In [0]:
# Material casting
datasets["materials"] = (
    datasets["materials"]
)
datasets["materials"].printSchema()

In [0]:
# Operating Hour casting
datasets["operating_hours"] = (
    datasets["operating_hours"]
)
datasets["operating_hours"].printSchema()

In [0]:
# Visit Plan casting
datasets["visit_plan"] = (
    datasets["visit_plan"]
)
datasets["visit_plan"].printSchema()

In [0]:
# Cutoff Time casting
datasets["cutoff_times"] = (
    datasets["cutoff_times"]
)
datasets["cutoff_times"].printSchema()

## Missing Value Handling

In [0]:
missing_values = missing_check(datasets)
display(missing_values)

In [0]:
datasets["google_analytics"] = (
    datasets["google_analytics"]
    .withColumn(
        "items",
        coalesce(
            col("items"),
            from_json(lit("[]"), items_schema),
        ),
    )
    .fillna({
        "device_mobile_brand_name": "Other",
        "event_page_name": "Unknown Page",
        "event_page_title": "Unknown Title",
    })
)

In [0]:
datasets["orders"] = (
    datasets["orders"]
    .dropna(subset=["material_id", "plant_id"])
)

In [0]:
datasets["customers"] = (
    datasets["customers"]
    .dropna(subset=["distribution_mode_description"])
)

In [0]:
datasets["materials"] = (
    datasets["materials"]
    .fillna({
        "bev_cat_desc": "Other",
    })
)

In [0]:
datasets["visit_plan"] = (
    datasets["visit_plan"]
)
quick_list(datasets["visit_plan"], "frequency")
quick_list(datasets["visit_plan"], "sales_office")
quick_list(datasets["visit_plan"], "sales_office_desc")
quick_list(datasets["visit_plan"], "distribution_mode")
quick_list(datasets["visit_plan"], "shipping_conditions_desc")

In [0]:
missing_values = missing_check(datasets)
display(missing_values)
del(missing_values)

## Outlier Detection

## Business Rule Enforement

# Feature Engineering

In [0]:
display(google_analytics.limit(5))

In [0]:
google_analytics = (
    google_analytics
)

google_analytics.printSchema()

In [0]:
display(orders.limit(5))

In [0]:
orders = (
    orders
)

orders.printSchema()

In [0]:
display(sales.limit(5))

In [0]:
sales = (
    sales
)

sales.printSchema()

In [0]:
display(customers.limit(5))

In [0]:
customers = (
    customers
)

customers.printSchema()

In [0]:
display(materials.limit(5))

In [0]:
materials = (
    materials
)

materials.printSchema()

In [0]:
display(operating_hours.limit(5))

In [0]:
operating_hours = (
    operating_hours
)

operating_hours.printSchema()

In [0]:
display(visit_plan.limit(5))

In [0]:
visit_plan = (
    visit_plan
)

visit_plan.printSchema()

In [0]:
display(cutoff_times.limit(5))

In [0]:
cutoff_times = (
    cutoff_times
)

cutoff_times.printSchema()

# Data Analysis

# Results